# Feast Feature Store on prokube

**Scenario:** You work at an online retailer. Your team wants to predict
whether a customer will return their next order. To do that, you need
customer-level features (order history, return rates, spending patterns)
available for both model training and real-time inference.

This notebook walks through the full Feast workflow:

1. **Setup** — install dependencies, configure the Feast client
2. **Generate data** — simulate customer order history
3. **Define features** — entities, feature views, and on-demand transformations
4. **Register** — push definitions to the Feast registry
5. **Train** — retrieve historical features, preprocess, train a return predictor
6. **Materialize** — push latest values to Redis for online serving
7. **Serve** — predict return risk for incoming orders in real time

Everything happens inline in this notebook — no external Python files or CLI
commands needed.

### Prerequisites

**Before running this notebook**, follow the setup steps in
[`README.md`](README.md) first. You need to create:

1. The **Redis password secret** (`redis-feast`)
2. The **Redis instance** (`redis-cr.yaml`) — wait until the pod is Running
3. The **Feast Redis secret** (`feast-redis-config`)
4. The **FeatureStore CR** (`feast-cr.yaml`) — wait until Ready

If any of these are missing, the notebook will fail at the "Configure the
Feast client" step.

> **Note:** This setup uses a SQLite registry on `/tmp` which does not
> survive pod restarts. For a production-ready deployment, see the
> [Production Setup](#Production-Setup) section at the end.


---
## 1. Setup


In [ ]:
!pip install -q 'feast[redis]' grpcio scikit-learn


### Configure the Feast client

Instead of building `feature_store.yaml` from scratch, we read the
operator-published `feast-<name>-client` **ConfigMap**. That file already
contains the right project name and a `registry_type: remote` pointing at
the registry gRPC server backed by a PVC — so feature definitions persist
across notebook restarts and are shared with everything else in the namespace.

We then override the `online_store` with a direct Redis connection (read
from the secret referenced by the FeatureStore CR) so that `materialize()`
can write feature values from this notebook. In a fully production setup,
materialization runs as a server-side CronJob and the notebook would keep
the remote online store config too — see the *Production Setup* section.


In [ ]:
import base64
import json
import subprocess

import yaml


def kubectl_json(*args):
    return json.loads(subprocess.check_output(["kubectl", *args, "-o", "json"]))


def get_namespace():
    """Read the current namespace from the pod's service account."""
    try:
        with open("/var/run/secrets/kubernetes.io/serviceaccount/namespace") as f:
            return f.read().strip()
    except FileNotFoundError:
        return subprocess.check_output(
            ["kubectl", "config", "view", "--minify", "-o", "jsonpath={..namespace}"]
        ).decode().strip()


# 1. Find the FeatureStore CR in this namespace.
fs_list = kubectl_json("get", "featurestore")["items"]
if not fs_list:
    raise RuntimeError("No FeatureStore CR found — apply feast-cr.yaml first.")
fs = fs_list[0]
fs_name = fs["metadata"]["name"]
client_cm_name = fs["status"]["clientConfigMap"]
redis_secret_name = (
    fs["spec"]["services"]["onlineStore"]["persistence"]["store"]["secretRef"]["name"]
)
redis_secret_key = (
    fs["spec"]["services"]["onlineStore"]["persistence"]["store"].get("secretKeyName", "redis")
)

# 2. Read the operator-published client config (project + remote registry).
#    feast-cr.yaml disables the istio sidecar on the feast-server pod
#    (sidecar.istio.io/inject: "false") so gRPC reaches the registry directly
#    via the operator's default Service.
client_cm = kubectl_json("get", "cm", client_cm_name)
config = yaml.safe_load(client_cm["data"]["feature_store.yaml"])

# 3. Read the Redis connection string from the secret referenced by the CR.
redis_secret = kubectl_json("get", "secret", redis_secret_name)
redis_yaml = base64.b64decode(redis_secret["data"][redis_secret_key]).decode()
redis_conn = yaml.safe_load(redis_yaml)["connection_string"]

# 4. Override online_store with a direct Redis connection so materialize()
#    works from this notebook. Add a local-file offline store for the parquet
#    we generate below.
config["online_store"] = {"type": "redis", "connection_string": redis_conn}
config["offline_store"] = {"type": "file"}

with open("feature_store.yaml", "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

NAMESPACE = get_namespace()
FEAST_PROJECT = config["project"]

print(f"FeatureStore CR:    {fs_name}")
print(f"Project:            {FEAST_PROJECT}")
print(f"Namespace:          {NAMESPACE}")
print(f"Registry (remote):  {config['registry']['path']}")
print(f"Online store:       redis @ {redis_conn.split(',')[0]}")
print("\nfeature_store.yaml written.")


---
## 2. Generate sample data

We simulate a customer order history table — the kind of data your data
pipeline would produce daily. Each row represents the aggregated stats for
one customer at one point in time:

| Column | Meaning |
|--------|--------|
| `customer_id` | Unique customer identifier |
| `total_orders` | Total number of orders placed |
| `total_returns` | Total number of returned orders |
| `avg_order_value` | Average order value in EUR |
| `days_since_last_order` | Days since the customer's last order |
| `returned` | Did the customer return their most recent order? (label) |

In a real project, this would come from your data warehouse or ETL pipeline.


In [ ]:
import datetime
import os

import numpy as np
import pandas as pd

np.random.seed(42)
n_customers = 200
n_snapshots = 10  # 10 daily snapshots per customer
n = n_customers * n_snapshots
now = datetime.datetime.now()

customer_ids = np.repeat(np.arange(1, n_customers + 1), n_snapshots)
timestamps = []
for _ in range(n_customers):
    timestamps.extend([now - datetime.timedelta(days=i) for i in range(n_snapshots)])

# Simulate realistic customer stats
total_orders = np.random.randint(1, 80, n).astype(np.int64)
total_returns = np.array([
    np.random.binomial(orders, np.random.uniform(0.05, 0.4))
    for orders in total_orders
]).astype(np.int64)
avg_order_value = np.random.uniform(15.0, 250.0, n).astype(np.float32)
days_since_last_order = np.random.randint(0, 90, n).astype(np.int64)

# Label: customers with high return rates and high order values are more
# likely to return. Add noise to keep it realistic.
return_rate = total_returns / np.maximum(total_orders, 1)
return_prob = 0.3 * return_rate + 0.002 * avg_order_value / 250.0 + np.random.normal(0, 0.1, n)
returned = (return_prob > 0.15).astype(np.int64)

customer_df = pd.DataFrame({
    "customer_id": customer_ids,
    "event_timestamp": timestamps,
    "total_orders": total_orders,
    "total_returns": total_returns,
    "avg_order_value": avg_order_value,
    "days_since_last_order": days_since_last_order,
    "returned": returned,
    "created": timestamps,
})

os.makedirs("data", exist_ok=True)
customer_df.to_parquet("data/customer_orders.parquet")
print(f"Created {n} rows for {n_customers} customers ({n_snapshots} snapshots each)")
print(f"Return rate in dataset: {returned.mean():.1%}")
customer_df.head(10)


---
## 3. Define features

In Feast, features are defined as Python objects:

- **Entity**: the primary key for lookups (here, `customer_id`).
- **DataSource**: where the raw feature data lives (parquet, table, etc.).
- **FeatureView**: declares which columns from the source are features and
  how long they're valid (`ttl`). Feast stores and serves them — but doesn't
  compute anything. Your data pipeline is responsible for producing the data.
- **FeatureService**: a named bundle of one or more feature views. Consumers
  (e.g. an inference API) reference the service by name instead of listing
  individual features — they don't need to know the internal view structure.

Derived features (`return_rate = total_returns / total_orders`, etc.) are
plain pandas computations in this notebook. Feast 0.63 also offers
*on-demand feature views* for serving-time transformations, but they don't
yet work reliably with the Operator-managed remote registry, so we skip
them here.


In [ ]:
from datetime import timedelta

import pandas as pd

from feast import Entity, FeatureService, FeatureStore, FeatureView, Field, FileSource, ValueType
from feast.on_demand_feature_view import on_demand_feature_view
from feast.types import Float32, Int64

# ---------------------------------------------------------------------------
# Entity: the "primary key" for feature lookups.
# When you request features, you provide a customer_id.
# ---------------------------------------------------------------------------
customer = Entity(
    name="customer_id",
    value_type=ValueType.INT64,
    description="Unique customer identifier",
)

# ---------------------------------------------------------------------------
# Data source: points to the parquet file produced by the data pipeline.
# ---------------------------------------------------------------------------
customer_orders_source = FileSource(
    path="data/customer_orders.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created",
)

# ---------------------------------------------------------------------------
# FeatureView: declares which columns from the source are features.
# These are raw/precomputed values from your data pipeline.
#
# Note: the source data also contains "returned" (did the customer return
# their last order?). We don't include it here because it's our prediction
# target — at serving time, we don't know the answer yet. Labels live
# outside Feast and are joined only during training.
# ---------------------------------------------------------------------------
customer_order_stats = FeatureView(
    name="customer_order_stats",
    entities=[customer],
    ttl=timedelta(days=30),
    schema=[
        Field(name="total_orders", dtype=Int64),
        Field(name="total_returns", dtype=Int64),
        Field(name="avg_order_value", dtype=Float32),
        Field(name="days_since_last_order", dtype=Int64),
    ],
    source=customer_orders_source,
    online=True,
)

# ---------------------------------------------------------------------------
# On-Demand Feature View: computes derived features at request time.
#
# Feast calls this function automatically during get_historical_features()
# and get_online_features() — no precomputation or manual pandas needed.
#
# Why on-demand rather than writing to the online store?
#   return_rate and return_risk are simple ratios. Computing them on-the-fly
#   is cheap, keeps Redis smaller, and avoids a known Feast ≤0.63 bug where
#   write_to_online_store=True fails with an entity-key serialization error
#   on certain Redis backends.
# ---------------------------------------------------------------------------
@on_demand_feature_view(
    sources=[customer_order_stats],
    schema=[
        Field(name="return_rate", dtype=Float32),
        Field(name="return_risk", dtype=Float32),
    ],
)
def customer_risk_features(inputs: pd.DataFrame) -> pd.DataFrame:
    df = pd.DataFrame()
    df["return_rate"] = (
        inputs["total_returns"] / inputs["total_orders"].clip(lower=1)
    ).astype("float32")
    df["return_risk"] = (df["return_rate"] * inputs["avg_order_value"]).astype("float32")
    return df


# ---------------------------------------------------------------------------
# Workaround: Feast ≤ 0.63 dill + typeguard hang on remote-registry ODFVs.
#
# When Feast fetches an ODFV from the remote gRPC registry it calls
# PandasTransformation.from_proto(), which calls dill.loads() to reconstruct
# the UDF. dill reconstructs the function's __globals__, re-importing feast
# modules and hitting typeguard's AST instrumentation — a deeply recursive
# traversal that hangs indefinitely.
#
# Fix: replace from_proto with a version that looks up the live function by
# name instead of deserializing via dill. Register every UDF defined in this
# session below.
# ---------------------------------------------------------------------------
from feast.transformation.pandas_transformation import PandasTransformation as _PT

_UDF_REGISTRY = {
    "customer_risk_features": customer_risk_features.feature_transformation.udf,
}

_orig_from_proto = _PT.from_proto.__func__

@classmethod
def _fast_from_proto(cls, proto):
    if proto.name in _UDF_REGISTRY:
        return cls(udf=_UDF_REGISTRY[proto.name], udf_string=proto.body_text)
    return _orig_from_proto(cls, proto)

_PT.from_proto = _fast_from_proto

# ---------------------------------------------------------------------------
# FeatureService: a named bundle of feature views.
#
# Instead of listing individual feature names in every get_online_features()
# call, consumers reference the service by name. This is especially useful
# when external services (e.g. an inference API) need features — they only
# need to know the service name, not the internal view structure.
# ---------------------------------------------------------------------------
customer_risk_service = FeatureService(
    name="customer_risk_service",
    features=[customer_order_stats, customer_risk_features],
)

print("Feature definitions created (not yet registered).")


---
## 4. Register features

`store.apply()` writes all definitions to the registry. After this call,
the feature views and service are visible to every other client in the
namespace — they persist on the operator-managed registry PVC.

You only need to re-run this cell if you change a definition.


In [ ]:
from feast import Project

store = FeatureStore(repo_path=".")

store.apply([
    Project(name=FEAST_PROJECT),
    customer,
    customer_orders_source,
    customer_order_stats,
    customer_risk_features,
    customer_risk_service,
])

print("Registered in the remote registry:")
for fv in store.list_feature_views():
    print(f"  FeatureView:    {fv.name}")
for odfv in store.list_on_demand_feature_views():
    print(f"  OnDemandFV:     {odfv.name}")
for fs in store.list_feature_services():
    print(f"  FeatureService: {fs.name}")


---
## 5. Retrieve historical features and train a model

`get_historical_features()` performs a **point-in-time join**: for each
customer, it finds the most recent feature values *as of that timestamp*.
This prevents data leakage — you only see features that were available when
the event occurred.

After Feast returns the raw features, we compute two derived columns
(`return_rate` and `return_risk`) in plain pandas. Keeping them outside
Feast keeps this example simple; in a real pipeline you'd either precompute
them upstream and add them to the FeatureView, or use an on-demand feature
view once the operator/feast-version combo supports it.


In [ ]:
# Build a query: "give me features for these customers, as of right now."
# Each row says: I want to know the feature values for customer X at time T.
# Feast will find the most recent feature values that were available at that
# timestamp — this is the "point-in-time join" that prevents data leakage.
#
# The FeatureService includes the on-demand feature view, so return_rate and
# return_risk are computed by Feast automatically as part of this call.

all_customer_ids = list(range(1, n_customers + 1))
query_timestamp = now  # "as of right now"

entity_df = pd.DataFrame({
    "customer_id": all_customer_ids,
    "event_timestamp": [query_timestamp] * len(all_customer_ids),
})

print(f"Querying features for {len(all_customer_ids)} customers...")

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=customer_risk_service,
).to_df()

# return_rate and return_risk are computed by the on-demand feature view —
# no manual pandas derivation needed here.

print(f"Retrieved {len(training_df)} rows.")
training_df.head(10)


### Preprocessing

Before training, we need to clean up the data:

1. **Join the label** — `returned` is our prediction target, not a feature.
   We deliberately keep it out of Feast because at serving time (when a
   customer places a new order) we don't know yet whether they will return
   it — that's what the model predicts. Labels typically come from a
   separate source (e.g. your data warehouse) and are joined only for
   training.
2. **Filter out new customers** — customers with fewer than 3 orders don't
   have enough history for reliable features. Feeding them into the model
   would add noise.
3. **Drop nulls** — any rows where Feast couldn't find matching features.
4. **Normalize** — scale numeric features so the model doesn't overweight
   high-magnitude columns like `avg_order_value`.


In [ ]:
from sklearn.preprocessing import StandardScaler

# --- 1. Join the label from the source data ---
# Get the most recent snapshot per customer to use as label
latest_labels = (
    customer_df
    .sort_values("event_timestamp")
    .groupby("customer_id")
    .last()[["returned"]]
    .reset_index()
)
training_df = training_df.merge(latest_labels, on="customer_id", how="left")

print(f"After joining labels: {len(training_df)} rows")

# --- 2. Filter out new customers (< 3 orders) ---
before = len(training_df)
training_df = training_df[training_df["total_orders"] >= 3].copy()
print(f"After filtering new customers (< 3 orders): {len(training_df)} rows (dropped {before - len(training_df)})")

# --- 3. Drop nulls ---
before = len(training_df)
training_df = training_df.dropna()
print(f"After dropping nulls: {len(training_df)} rows (dropped {before - len(training_df)})")

# --- 4. Normalize features ---
FEATURE_COLS = ["total_orders", "total_returns", "avg_order_value",
                "days_since_last_order", "return_rate", "return_risk"]
TARGET = "returned"

scaler = StandardScaler()
X = pd.DataFrame(
    scaler.fit_transform(training_df[FEATURE_COLS]),
    columns=FEATURE_COLS,
    index=training_df.index,
)
y = training_df[TARGET]

print(f"\nTraining set: {len(X)} samples, {len(FEATURE_COLS)} features")
print(f"Class balance: {y.mean():.1%} returns")
X.describe().round(2)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["kept", "returned"]))


---
## 6. Materialize features to Redis

Materialization copies the latest feature values from the offline store
(parquet) into Redis for low-latency online serving.

Note: only `FeatureView` data is materialized to Redis. The ODFV features
(`return_rate`, `return_risk`) are computed on-the-fly when you call
`get_online_features()` — Feast reads the raw values from Redis and applies
the transformation inline.

In production you would run this on a schedule (e.g. daily after your
pipeline updates the parquet files).


In [ ]:
from datetime import datetime, timedelta

store.materialize(
    start_date=datetime.now() - timedelta(days=30),
    end_date=datetime.now(),
)
print("Materialized to Redis.")


---
## 7. Online feature serving — predict return risk

A customer just placed an order. Your order service calls Feast to get
their features from Redis, computes the same derived columns we used in
training, feeds the result into the model, and decides whether to flag the
order for proactive customer service.


In [ ]:
# Simulate: these 3 customers just placed a new order
customers_with_new_orders = [{"customer_id": 5}, {"customer_id": 42}, {"customer_id": 137}]

# Use the FeatureService instead of listing individual features.
# The service bundles all views — consumers don't need to know the internals.
# The on-demand feature view (return_rate, return_risk) is applied automatically
# by the SDK at request time before returning results.
online_features = store.get_online_features(
    features=customer_risk_service,
    entity_rows=customers_with_new_orders,
).to_dict()

online_df = pd.DataFrame(online_features)

print("Online features (raw from Redis + on-demand derived columns):")
online_df


In [ ]:
# Run inference: predict return probability and flag high-risk orders
X_inference = pd.DataFrame(
    scaler.transform(online_df[FEATURE_COLS]),
    columns=FEATURE_COLS,
)

return_probabilities = model.predict_proba(X_inference)[:, 1]

RISK_THRESHOLD = 0.5

print("\n--- Return Risk Assessment ---")
for cid, prob in zip(online_df["customer_id"], return_probabilities):
    flag = "HIGH RISK" if prob > RISK_THRESHOLD else "low risk"
    print(f"  Customer {cid}: return probability = {prob:.1%} [{flag}]")


---
## Summary

| Step | API | What happens |
|------|-----|-------------|
| Define | Python objects (Entity, FeatureView, FeatureService) | Declare what features exist and how they are grouped |
| Register | `store.apply([...])` | Write definitions to the **remote registry** served by the Feast Operator |
| Train | `store.get_historical_features(features=service)` | Point-in-time join from parquet |
| Preprocess | pandas / sklearn | Derive columns, filter, clean, normalize |
| Materialize | `store.materialize()` | Push latest raw values to Redis |
| Serve | `store.get_online_features(features=service)` | Sub-ms lookup from Redis |

### FeatureService

A `FeatureService` bundles one or more feature views under a single name.
Clients (notebooks, inference services) reference the service name instead
of listing individual feature names — they don't need to know the internal
view structure. Define once, use everywhere.

### About the registry

The registry is the gRPC service deployed by the Feast Operator
(`feast-<name>-registry`) and backed by the persistent volume on the
FeatureStore CR. Feature *definitions* you `apply()` from this notebook
are visible to every other client in the namespace and survive pod
restarts. Feature *data* lives in Redis (online) and parquet on the PVC
(offline).


---
## Production hardening

This notebook already uses the production registry served by the Feast
Operator. To make the rest of the workflow production-grade:

### 1. Feature definitions live in Git

Instead of defining features in a notebook, put them in a `features.py`
file in a Git repository. The Feast Operator clones the repo on startup
and runs `feast apply` automatically:

```yaml
# feast-cr.yaml
spec:
  feastProject: retail_features
  feastProjectDir:
    git:
      url: https://github.com/your-org/feast-feature-repo
      ref: main  # or pin to a commit SHA
```

Feature definitions are then version-controlled, reviewed via PRs, and
automatically deployed when the pod starts. Notebooks shift from being
authors of definitions to consumers of them.

### 2. Materialization runs as a CronJob

Instead of running `store.materialize()` from a notebook, set up a
Kubernetes CronJob that runs on a schedule (e.g. daily, after your ETL
pipeline refreshes the customer data). The Feast Operator can manage this
via the `batchEngine` config.

### 3. Use a SQL-backed registry

Switch the registry persistence from PVC-backed SQLite to PostgreSQL —
better for multi-replica feast-server deployments and concurrent writes.

```yaml
spec:
  services:
    registry:
      local:
        server: {}
        persistence:
          store:
            type: sql
            secretRef:
              name: feast-registry-db
```

### 4. Component overview

| Component | This notebook | Production |
|-----------|---------------|------------|
| Registry | gRPC server, PVC-backed SQLite | gRPC server, PostgreSQL |
| Online Store | Redis (operator-managed) | Same |
| Offline Store | Parquet on PVC | Parquet on PVC, or S3/MinIO |
| Feature definitions | Defined in notebook | Defined in Git, applied on operator startup |
| Materialization | Run from notebook | CronJob |
| Feast Server | 1 replica | Multi-replica with HPA |

For the full production deployment guide, see the
[Feast Production Deployment Topologies](https://docs.feast.dev/how-to-guides/production-deployment-topologies)
documentation.
